# Objectif du notebook :
Pouvoir étudier la jeopardysation de MQC

# Configuration

## Imports

In [ ]:
import requests
from openai import OpenAI
from getpass import getpass
from IPython.display import Markdown, display
from datetime import datetime
import pandas as pd

## Initialisation de la session HTTP

In [ ]:
base_url = "https://albert.api.etalab.gouv.fr/v1"
api_key = getpass("Clef d’API Albert: ")
client = OpenAI(base_url=base_url, api_key=api_key)
session = requests.session()
session.headers = {"Authorization": f"Bearer {api_key}"}

In [ ]:
identifiant_collection_indexation = ""
identifiant_collection_jeopardy = ""

# Récupérer une collection

In [ ]:
def get_collection(collection_id):
    response = session.get(f"{base_url}/collections/{collection_id}")
    response = response.json()
    return response


collection_indexation = get_collection(identifiant_collection_indexation)
collection_jeopardy = get_collection(identifiant_collection_jeopardy)
nombre_de_documents_indexation = collection_indexation.get("documents", 0)
nombre_de_documents_jeopardy = collection_jeopardy.get("documents", 0)

display(
    Markdown(
        "## Indexation\n"
        f"### Collection - {collection_indexation['name']} (créée le {datetime.fromtimestamp(collection_indexation['created']).strftime('%d/%m/%Y %H:%M:%S')})\n"
        f"- **Description :** {collection_indexation['description']}\n"
        f"- **Nombre de documents :** {nombre_de_documents_indexation}\n"
        "## Jeopardy\n"
        f"### Collection - {collection_jeopardy['name']} (créée le {datetime.fromtimestamp(collection_jeopardy['created']).strftime('%d/%m/%Y %H:%M:%S')})\n"
        f"- **Description :** {collection_jeopardy['description']}\n"
        f"- **Nombre de documents :** {nombre_de_documents_jeopardy}\n"
    )
)

### Lister les documents

In [ ]:
def liste_documents_dans_collection(collection_id, nombre_de_documents):
    resultats = []

    for offset in range(0, nombre_de_documents, 100):
        params = {
            "collection_id": collection_id,
            "limit": 100,
            "offset": offset,
        }
        response = session.get(f"{base_url}/documents", params=params)
        resultats.extend(response.json()["data"])
    return resultats


les_documents_indexation = liste_documents_dans_collection(identifiant_collection_indexation, nombre_de_documents_indexation)
les_documents_jeopardy = liste_documents_dans_collection(identifiant_collection_jeopardy, nombre_de_documents_jeopardy)

### Indexation

In [ ]:
dataframe_indexation = pd.DataFrame(les_documents_indexation)
dataframe_indexation

### Jeopardy

In [ ]:
dataframe_jeopardy = pd.DataFrame(les_documents_jeopardy)
dataframe_jeopardy

### Lister les différences entre l’indexation et jeopardy

In [ ]:
df_diff = dataframe_indexation.merge(dataframe_jeopardy, on='name', how='outer', indicator=True)

documents_dans_indexation_uniquement = df_diff[df_diff['_merge'] == 'left_only']
documents_dans_jeopardy_uniquement = df_diff[df_diff['_merge'] == 'right_only']
documents_dans_indexation_uniquement